# Different API keys

1. **GROQ**
2. OpenRouter
3. OpenAI
4. Anthropic

In [29]:
import json
import os
import time
import json
import logging
import requests
from dotenv import load_dotenv

load_dotenv()

True

# SDK vs. Raw HTTP: A Comparison

1. Typed responses
2. Streaming
3. Typed exceptions
4. Retries with backoff
5. Connection pooling
6. Versioning

## Groq

In [30]:
import groq
from groq import Groq

GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or "PASTE_YOUR_KEY_HERE"
MODEL = "openai/gpt-oss-20b"
BASE_URL = "https://api.groq.com/openai/v1/chat/completions"

assert GROQ_API_KEY and GROQ_API_KEY != "PASTE_YOUR_KEY_HERE", (
    "Set GROQ_API_KEY as an environment variable, or paste your key above. "
    "Get a free one at https://console.groq.com/keys")

The raw request without using SDK.

In [31]:
headers = {"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"}

The SDK client we will compare against.

In [32]:
client = Groq(api_key=GROQ_API_KEY)
print("Ready.")

Ready.


# SDK vs. Raw HTTP: A Comparison

1. Typed responses
2. Streaming
3. Typed exceptions
4. Retries with backoff
5. Connection pooling
6. Versioning

# The question we will ask to the LLM

In [33]:
payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "How many r's are in the word 'strawberry'?"}],
}

## 1. Typed Responses

**Without an SDK**, a response is just a dictionary. You have to know
the exact nested shape and type every key correctly.

**With an SDK**, the response is a typed object. Your editor can
autocomplete `.choices[0].message.content`, and a typo is caught by
your editor, not by a runtime crash in front of users.

In [43]:
# ----- WITHOUT the SDK: raw requests.post + manual dict parsing -----
raw_response = requests.post(BASE_URL, headers=headers, json=payload)
print("Type of response:", type(raw_response.text).__name__)
print("--"*5)
print("Raw response:", raw_response.text)

Type of response: str
----------
Raw response: {"id":"chatcmpl-60774141-91ec-440d-87e6-d87d466814a0","object":"chat.completion","created":1789793523,"model":"openai/gpt-oss-20b","choices":[{"index":0,"message":{"role":"assistant","content":"There are **3** 'r' letters in the word “strawberry.”","reasoning":"We need to answer the question: \"How many r's are in the word 'strawberry'?\" The word 'strawberry' spelled s-t-r-a-w-b-e-r-r-y. Count r's: there are r at position 3, r at position 8, r at position 9? Let's count: s(1), t(2), r(3), a(4), w(5), b(6), e(7), r(8), r(9), y(10). So r appears at 3, 8, 9. So 3 r's. But some people might think \"strawberry\" has 2 r's? Wait, let's check: the spelling \"strawberry\" actually has 2 r's? Let's verify. The word strawberry: s t r a w b e r r y. Yes, 3 r's: r at 3, r at 8, r at 9. So answer: 3. Should we mention that? Probably answer: 3. The user just asked the question. We respond succinctly."},"logprobs":null,"finish_reason":"stop"}],"usage":{

In [44]:
# ----- WITH the SDK: a typed response object -----
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "How many r's are in the word 'strawberry'?"}],
)

print("Type of response:", type(response).__name__)
print("--"*5)

print("Message content: ", response.choices[0].message.content)
# response.choices[0].message.<TAB> would autocomplete in your editor

Type of response: ChatCompletion
----------
Message content:  There are **3** r's in "strawberry."


## 2. Streaming

**Without an SDK**, a streamed response arrives as raw
server-sent-event lines. You must strip the `data: ` prefix, detect
the `[DONE]` sentinel, and `json.loads` every chunk yourself.

**With an SDK**, you just iterate over already-parsed chunk objects.

You have seen response like this using ChatGPT or similar AI Assistants.

#### Will see in Part 2 demo

## 3. Typed Exceptions

To simulate this, we are using a **fake** API key for Groq: gsk_NsB5GIJbXnH9SH12CJ1WGnanb3FYzWETlWrbyH9OBNas45jfEkFNFy

**Without an SDK**, you check `status_code` yourself and re-derive
"is this worth retrying?" every single time you call the API.

**With an SDK**, you catch a named exception that already encodes
that judgment.

In [36]:
fake_api_key = "gsk_NsB5GIJbjdnejXnH9SH12CJ1WGnanb3FYzWETlWrbyH9OBNas45jfEkFNFy"

In [45]:
# ----- WITHOUT the SDK: check status codes yourself -----
header = {"Authorization": fake_api_key , "Content-Type": "application/json"}
raw_response = requests.post(BASE_URL, headers=header, json=payload)

if raw_response.status_code == 401:
    print(f"Got HTTP {raw_response.status_code}: Bad API key.")
elif raw_response.status_code == 429:
    print(f"Got HTTP {raw_response.status_code}: rate limited. Worth retrying.")
elif raw_response.status_code >= 500:
    print(f"Got HTTP {raw_response.status_code}: server error. Worth retrying.")
else:
    print(f"Got HTTP {raw_response.status_code}: {raw_response.text[:200]}")

Got HTTP 401: Bad API key.


In [46]:
# ----- WITH the SDK: catch a specific, named exception -----
bad_client = Groq(api_key=fake_api_key)
try:
    bad_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "This call will fail."}],
    )
except groq.AuthenticationError as e:
    print(f"Caught {type(e).__name__} -> bad/missing API key. Not worth retrying.")
except groq.RateLimitError as e:
    print(f"Caught {type(e).__name__} -> rate limited. Worth retrying.")
except groq.APIStatusError as e:
    print(f"Caught {type(e).__name__} -> HTTP {e.status_code}")

Caught AuthenticationError -> bad/missing API key. Not worth retrying.


## 4. Retries with Exponential Backoff

**Without an SDK**, you hand-roll a retry loop — and a naive one
retries on *every* non-200 response, including a `400 bad request`,
which will just fail identically every time and waste your retry
budget.

```python
for i in range(10):
    response = requests.post(url)
    if response.status_code == 200:
        break
```
But the above retries 10 times immediately.

**Exponential Backoff**

You might have seen this on putting wrong password to the phone.

```
    Attempt 1 → fails
    wait 1 second
    Attempt 2 → fails
    wait 2 seconds
    Attempt 3 → fails
    wait 4 seconds
    Attempt 4 → fails
    wait 8 seconds
```
**With an SDK**, retryable vs. non-retryable errors are already distinguished for you, with backoff built in.

```
    429 → retry
    500 → retry
    400 → don't retry
    401 → don't retry
```
A `400` usually means there is something wrong with our request. Retrying the exact same request won't fix it.

## 5. Connection Pooling

**Without an SDK**, a plain `requests.post()` call with no `Session`
object does not reuse the underlying connection between calls.

```
Create TCP connection
        ↓
TLS handshake
        ↓
Send request
        ↓
Receive response
        ↓
Close connection
```

**With an SDK**, one client keeps a connection pool alive under the
hood (via `httpx`), so only the first call typically pays for the
TCP+TLS handshake.

```
Create connection
        ↓
TLS handshake
        ↓
       Pool
    ↙   ↓   ↘
Request Request Request
```

The time printed could be misleading.

In [47]:
# ----- WITHOUT the SDK: a fresh connection for every call -----
raw_timings = []
for i in range(3):
    start = time.time()
    requests.post(BASE_URL, headers=headers, json={
        "model": MODEL,
        "messages": [{"role": "user", "content": f"reply with just the digit {i}"}],
    })
    raw_timings.append(time.time() - start)

for i, t in enumerate(raw_timings):
    print(f"Raw request {i + 1}: {t:.3f}s (new connection each time)")

Raw request 1: 0.610s (new connection each time)
Raw request 2: 0.598s (new connection each time)
Raw request 3: 0.616s (new connection each time)


In [48]:
# ----- WITH the SDK: one client, one reused connection pool -----
sdk_timings = []
for i in range(3):
    start = time.time()
    client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"reply with just the digit {i}"}],
    )
    sdk_timings.append(time.time() - start)

for i, t in enumerate(sdk_timings):
    label = "(new  connection)" if i == 0 else "(reuses open connection)"
    print(f"SDK request {i + 1}: {t:.3f}s {label}")

SDK request 1: 0.675s (new  connection)
SDK request 2: 0.368s (reuses open connection)
SDK request 3: 0.446s (reuses open connection)


## 6. Versioning

**Without an SDK**, your parsing code is brittle to API changes — an added field could break in production.

**With an SDK**, you generally just run `pip install --upgrade` and get the update.

In [41]:
# ----- WITHOUT the SDK: your parsing code is brittle to API changes -----
def parse_raw_response_v1(data):
    # written against current API shape
    return data["choices"][0]["message"]["content"]

# Suppose the API changes "content" to "text" -- it breaks (KeyError)
renamed_api_response = {
    "choices": [{"message": {"role": "assistant", "text": "Hi there!"}}],
}
# You do many of checks.
choices = renamed_api_response.get("choices") or []
if choices:
    message = choices[0].get("message")
    if message:
        content = message.get("content", "")

SDK maintainers can map multiple names to the same object attribute using aliases. Here 'content' and 'text' mean the same. So, your code does not need to change.

In [42]:
# ----- The SDK maintainer's code -----
from pydantic import BaseModel, Field
class Message(BaseModel):
    content: str | None = Field(default=None, alias="text")

## Wrap-up

None of the raw-HTTP versions above are *impossible* — they're maybe
10-30 lines each. The point is that every project calling this API
ends up re-writing (and re-debugging) that same 10-30 lines. The SDK
is that work, done once, tested, and maintained for you.